In [14]:
import os
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from openai import AzureOpenAI
from config import (
    AZURE_OPENAI_API_KEY,      # <--- Falta esto
    AZURE_OPENAI_ENDPOINT,     # <--- Y esto
    EMBEDDING_MODEL,
    QDRANT_URL,
    QDRANT_API_KEY,
    COLLECTION_NAME,
)




In [15]:
# Conexión
client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY
)
print("✅ Conectado a Qdrant")



# 2. Conexión a Azure OpenAI (En lugar de cargar el modelo localmente)
print(f"🌐 Conectando con Azure OpenAI para embeddings...")
ai_client = AzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2024-02-01",
    azure_endpoint=AZURE_OPENAI_ENDPOINT
)

# 3. Definición del tamaño del vector 
# (Para text-embedding-3-small es 1536, para large puede ser 3072)
VECTOR_SIZE = 1536 
print(f"✅ Pipeline listo. Tamaño del vector esperado: {VECTOR_SIZE}")

✅ Conectado a Qdrant
🌐 Conectando con Azure OpenAI para embeddings...
✅ Pipeline listo. Tamaño del vector esperado: 1536


In [ ]:
#Creación de la colección

client.create_collection(
    collection_name="Info_LEGAL_AZURE",  # <--- Agregadas comillas
    vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE),
)
    
print(f"✅ Colección  creada exitosamente.")

✅ Colección  creada exitosamente.


In [16]:
DATA_DIR = "./data_LEGAL"


In [17]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
print(f"📂 Leyendo PDFs desde: {DATA_DIR}")
# Carga todos los PDFs de la carpeta
loader = PyPDFDirectoryLoader(DATA_DIR)
documents = loader.load()
print(f"📄 Se cargaron {len(documents)} páginas en total.")

# Configurar el chunking
# Un chunk de 1000 caracteres suele equivaler a unos 200-250 tokens
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150, # Superposición para mantener contexto
    length_function=len,
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = text_splitter.split_documents(documents)
print(f"✂️ Se generaron {len(chunks)} chunks (fragmentos) a partir de los PDFs.")

# Inspeccionar el primer chunk para verificar
if chunks:
    print("\n👀 --- MUESTRA DEL PRIMER CHUNK ---")
    print(f"Texto: {chunks[0].page_content[:200]}...")
    print(f"Metadatos: {chunks[0].metadata}")

📂 Leyendo PDFs desde: ./data_LEGAL
📄 Se cargaron 11 páginas en total.
✂️ Se generaron 34 chunks (fragmentos) a partir de los PDFs.

👀 --- MUESTRA DEL PRIMER CHUNK ---
Texto: Los compradores tendrán derecho a la garan�a legal, de conformidad con lo establecido en la 
Ley 1480 de 2011 (Estatuto del Consumidor), la cual dispone que: 
Artículo 7°. Garantía legal. Es la obliga...
Metadatos: {'producer': 'Adobe PDF Library 20.6.74', 'creator': 'Acrobat PDFMaker 20 para Word', 'creationdate': '2026-01-08T21:03:09-05:00', 'author': 'usuario', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2026-01-08T21:03:44-05:00', 'sourcemodified': 'D:20260109020246', 'subject': '', 'title': '', 'source': 'data_LEGAL\\ecm1.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}


In [18]:
from qdrant_client.models import PointStruct
import uuid
import time

# Configuración de rendimiento
BATCH_SIZE = 50 
points = []

print(f"⚙️ Generando vectores con Azure OpenAI ({EMBEDDING_MODEL})...")

for chunk in chunks:
    try:
        # 1. Generar vector usando la API de Azure
        # Importante: model=EMBEDDING_MODEL debe ser el nombre del 'Deployment' en Azure
        embedding_response = ai_client.embeddings.create(
            input=[chunk.page_content.replace("\n", " ")],
            model=EMBEDDING_MODEL 
        )
        vector = embedding_response.data[0].embedding
        
        # 2. Construir el payload
        payload = {
            "source": chunk.metadata.get("source", "unknown"),
            "page": chunk.metadata.get("page", 0),
            "text": chunk.page_content, 
            "document_type": "pdf_general"
        }
        
        # 3. Preparar el punto para Qdrant
        points.append(
            PointStruct(
                id=str(uuid.uuid4()),
                vector=vector,
                payload=payload
            )
        )
        
        # Opcional: Pequeño delay si tienes un Tier de Azure con Rate Limit muy bajo
        # time.sleep(0.01) 

    except Exception as e:
        print(f"❌ Error generando embedding para un chunk: {e}")
        continue

# --- Subida a Qdrant ---
print(f"\n🚀 Subiendo {len(points)} vectores a Qdrant en batches de {BATCH_SIZE}...")

for i in range(0, len(points), BATCH_SIZE):
    batch = points[i:i+BATCH_SIZE]
    print(f"📦 Upserting Batch {i//BATCH_SIZE + 1}...")
    client.upsert(
        collection_name="Info_Legal_AZURE",
        points=batch
    )

print("\n🎉 INGESTIÓN DE PDFS COMPLETADA EXITOSAMENTE")

⚙️ Generando vectores con Azure OpenAI (text-embedding-3-small)...

🚀 Subiendo 34 vectores a Qdrant en batches de 50...
📦 Upserting Batch 1...

🎉 INGESTIÓN DE PDFS COMPLETADA EXITOSAMENTE
